# 10장 보안 실습 — Journal·Audit 이벤트 연결


## Goal

서비스의 부팅 문맥과 Audit 레코드·이벤트 단위를 구분합니다.

[교안과 분석 질문](../../10-program-architecture/10-4-audit-analysis.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-10-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'journal-review.psv': 'time_kst|boot_id|unit|pid|message\n2026-09-10T08:00:00+09:00|BOOT-A|init.scope|1|System boot\n2026-09-10T09:02:00+09:00|BOOT-A|ssh.service|104|Accepted publickey for analyst\n2026-09-10T09:05:00+09:00|BOOT-A|report-helper.service|1|Started report helper\n2026-09-10T09:06:00+09:00|BOOT-A|report-helper.service|520|Report completed\n', 'audit.log': 'type=SYSCALL msg=audit(1788998580.000:900): arch=c000003e syscall=59 success=yes exit=0 a0=0 a1=0 a2=0 a3=0 items=1 ppid=410 pid=450 auid=1000 uid=0 gid=0 euid=0 suid=0 fsuid=0 egid=0 sgid=0 fsgid=0 tty=pts0 ses=4 comm="id" exe="/usr/bin/id" key="course_exec"\ntype=EXECVE msg=audit(1788998580.000:900): argc=1 a0="id"\ntype=CWD msg=audit(1788998580.000:900): cwd="/home/analyst"\ntype=PATH msg=audit(1788998580.000:900): item=0 name="/usr/bin/id" inode=100 dev=08:01 mode=0100755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL\ntype=PROCTITLE msg=audit(1788998580.000:900): proctitle=6964\ntype=EOE msg=audit(1788998580.000:900):\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 서비스 2행, Audit 6레코드/1이벤트, auid1000/euid0은 승인 여부와 별개

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. Journal 요약에서 서비스와 부팅 문맥 선택


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $3=="report-helper.service" {print $1 "|" $2 "|" $5}' \
 "$COURSE_DATA/journal-review.psv" > "$COURSE_OUT/service-events.psv"
test "$(wc -l < "$COURSE_OUT/service-events.psv")" -eq 2
cat "$COURSE_OUT/service-events.psv"


### 2. Audit 레코드와 이벤트 수 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
sed -n 's/.*msg=audit(\([^)]*\)).*/\1/p' "$COURSE_DATA/audit.log" | LC_ALL=C sort -u > "$COURSE_OUT/event-ids.txt"
test "$(wc -l < "$COURSE_DATA/audit.log")" -eq 6
test "$(wc -l < "$COURSE_OUT/event-ids.txt")" -eq 1
grep -Fx '1788998580.000:900' "$COURSE_OUT/event-ids.txt"
printf 'audit_records=6 audit_events=1\n'


### 3. 원래 로그인 ID와 실행 권한 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'type=SYSCALL ' "$COURSE_DATA/audit.log" > "$COURSE_OUT/syscall.txt"
grep -qF 'auid=1000 uid=0 gid=0 euid=0' "$COURSE_OUT/syscall.txt"
grep -qF 'exe="/usr/bin/id"' "$COURSE_OUT/syscall.txt"
printf 'auid=1000 euid=0 command=id approval=not_in_audit_record\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: euid=0은 무엇을 입증하는가

**Red Team 질문:** 정책상 위임한 권한과 관찰된 실행 권한이 일치하는가? GTFOBins에서 검토한 기능·권한 문맥을 실제 실행 기록에 연결할 때도 승인 여부는 별도의 자료가 필요합니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 위임된 권한 사용과 업무 범위의 차이 검토. UID 0 실행 자체가 불법이라는 가정 금지 |
| Command / Observation | ausearch·aureport 또는 합성 형식 집계로 이벤트 900의 6레코드를 하나로 묶음 |
| System Change / Artifact | /usr/bin/id의 실행·주체·인수·경로. 이 이벤트만으로 계정 파일 수정은 입증되지 않음 |
| Log prerequisite | 사건 시점 Audit 규칙·수집 범위·손실 상태. 뒤늦은 규칙 추가로 과거 행위가 복원되지 않음 |
| Blue Team Investigation | auid1000·euid0·부모·시각을 sudo 기록 및 승인 작업과 비교. 숫자 ID 보존 |
| Detection | 주체·실행 권한·인수·허용 업무의 차이를 검토. 단순 euid0 필터는 정상 관리자 작업도 선택 |
| Mitigation | 위임 범위 재검토·필요한 이벤트 수집·보존·주체 추적 개선. 광범위 기록의 민감정보·부하 평가 |

**반례와 해설:** 09:03 sudo 로그에 id 실행이 있고 같은 시각 Audit 이벤트가 있습니다. 서로 부합하는 실행 근거지만 승인 문서가 없으므로 정당성은 미확인입니다. sudo 한 행과 Audit 여섯 행을 일곱 번의 권한 상승으로 세면 중복·의미 오류가 함께 발생합니다.

**제출 과제:** 원문 이벤트 식별자, 주체, 유효 권한, 실행 경로를 적고 “실행 확인 / 승인 미확인 / 영향 미확인”을 구분합니다. 정상 id 실행에도 해당하는 탐지 조건을 제시하고 어떤 추가 문맥으로 오탐을 줄일지 설명합니다.


## Red Team ↔ Blue Team 사례 분석

### 사례: 서비스 시작 기록만으로 행위를 설명할 수 있는가

**Red Team 질문:** 서비스 실행 주체와 구성 변경 권한 사이의 경계가 적절한가? 서비스 시작 기록은 평가 대상 동작을 이해하는 자료이지만 누가 어떤 파일을 변경했는지까지 보여주지는 않습니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 서비스 기능과 권한 경계 검토. 구성 수정 권한과 서비스 실행 권한은 별도 |
| Command / Observation | journalctl의 기간·부팅·unit 필터, 합성 BOOT-A의 서비스 두 행 검토 |
| System Change / Artifact | 시작·중지·실패 메시지와 부팅 문맥. 파일 변경이나 모든 자식 프로세스 실행이 자동 포함되지는 않음 |
| Log prerequisite | Journal 저장 방식·보존·권한·시간 동기화·수집 범위 확인 |
| Blue Team Investigation | 동일 부팅의 서비스 메시지와 unit/drop-in 사본·변경 이력·프로세스 시작 비교 |
| Detection | 예상 밖 unit·실행 주체·시각의 조합을 검토하고 승인 재시작을 정상 사례로 테스트 |
| Mitigation | 서비스 권한·구성 변경 관리와 Journal 보존을 개선. 조사 중 임의 restart는 피함 |

**반례와 해설:** 패치 후 정상 재시작도 start/stop 이벤트를 남깁니다. 서로 다른 부팅의 같은 서비스 이름을 하나의 실행으로 연결하면 시간순 해석이 잘못됩니다. 현재 활성 unit과 사건 당시 unit 사본이 같은 버전인지도 확인해야 합니다.

**제출 과제:** BOOT-A의 두 행에서 확인되는 사실과 확인되지 않는 변경 주체를 분리합니다. 서비스 로그 필터를 좁히면서 빠질 수 있는 인증·Audit 근거를 적습니다. 보존 범위 밖의 기록을 “해당 행위 없음”으로 보고하지 않으면 통과입니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
